In [8]:
#19.Define a parameter grid for your classifier, for example the number of trees and 
#the maximum depth of a Random Forest, written using the double underscore pipeline naming.
import seaborn as sb
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
titanic = sb.load_dataset("titanic")
X = titanic.drop(["survived", "alive"], axis=1)
y = titanic["survived"]
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
numerical_columns = X.select_dtypes(include=["int64", "float64"]).columns
categorical_columns = X.select_dtypes(include=["object", "string", "category", "bool"]).columns
numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")),("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),("encoder", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([("numeric", numeric_pipeline, numerical_columns),("categorical", categorical_pipeline, categorical_columns)])
model_pipeline=Pipeline([("preprocessor", preprocessor),("classifier", RandomForestClassifier(random_state=42))])
param_grid = { "classifier__n_estimators": [50, 100, 200],"classifier__max_depth": [5, 10, 15, None]}


In [14]:
#20.Run GridSearchCV with five fold cross validation and print both the best parameters and
#the best cross validated score.
from sklearn.model_selection import GridSearchCV
import time
grid_search=GridSearchCV(model_pipeline,param_grid,cv=5,scoring="accuracy")
start_time = time.time()
grid_search.fit(X_train, y_train)
grid_time = time.time() - start_time
print("Best Parameters:", grid_search.best_params_)
print("Best CV Score:", grid_search.best_score_)
print("Grid Search Time:", grid_time, "seconds")

Best Parameters: {'classifier__max_depth': 5, 'classifier__n_estimators': 50}
Best CV Score: 0.8217078695951935
Grid Search Time: 35.97997164726257 seconds


In [15]:
#21.Run RandomizedSearchCV over wider ranges with a fixed number of iterations,
#and compare the result and the time taken against the grid search.
from sklearn.model_selection import RandomizedSearchCV
random_param_grid = {
    "classifier__n_estimators": [50, 100, 150, 200, 250, 300],
    "classifier__max_depth": [5, 10, 15, 20, 25, None]}
randomized_search=RandomizedSearchCV(model_pipeline,random_param_grid,n_iter=10, random_state=42,
    n_jobs=-1,cv=5,scoring="accuracy")
randomized_start_time = time.time()
randomized_search.fit(X_train,y_train)
randomized_time = time.time() - start_time
print("Best Parameters:", randomized_search.best_params_)
print("Best CV Score:", randomized_search.best_score_)
print("Randomized Search Time:", randomized_time, "seconds")

Best Parameters: {'classifier__n_estimators': 150, 'classifier__max_depth': 10}
Best CV Score: 0.8104895104895105
Randomized Search Time: 78.25229907035828 seconds


In [16]:
#22.Include at least one preprocessing choice in the search, such as the imputation strategy,
#to prove to yourself that the whole pipeline really is tunable.
from sklearn.model_selection import GridSearchCV
param_grid = {
    "classifier__n_estimators": [50, 100, 150],
    "classifier__max_depth": [5, 10, 15],
    "preprocessor__numeric__imputer__strategy": ["mean", "median"]
}
grid_search = GridSearchCV(model_pipeline,param_grid,cv=5,scoring="accuracy",n_jobs=-1)
grid_search.fit(X_train, y_train)
print("Best Parameters:", grid_search.best_params_)
print("Best CV Score:", grid_search.best_score_)

Best Parameters: {'classifier__max_depth': 5, 'classifier__n_estimators': 150, 'preprocessor__numeric__imputer__strategy': 'mean'}
Best CV Score: 0.8259332217078696


In [18]:
#23.Score the single best model once on your untouched test set, and 
#note how that number compares to the cross validated score the search reported
from sklearn.metrics import accuracy_score
best_model = grid_search.best_estimator_
y_pred=best_model.predict(X_test)
test_score = accuracy_score(y_test, y_pred)
print("Best CV Score:", grid_search.best_score_)
print("Final Test Score:", test_score)


Best CV Score: 0.8259332217078696
Final Test Score: 0.8212290502793296
